# Chapter 14 - Dimensionality Reduction

Real-world datasets often have dozens or hundreds of features. Many of those features
are correlated, redundant, or noisy. **Dimensionality reduction** compresses the data
into fewer dimensions while preserving as much useful information as possible.

The two most important techniques are:
- **PCA** (Principal Component Analysis) — a linear method that finds the directions
  of maximum variance.
- **t-SNE** — a nonlinear method optimised for 2D/3D visualisation of
  high-dimensional structure.

**Topics covered:**
- The curse of dimensionality
- PCA: mathematical intuition and sklearn usage
- Explained variance ratio and scree plots
- PCA for visualisation (Iris 4D to 2D)
- PCA for noise reduction
- t-SNE for nonlinear visualisation
- PCA vs t-SNE comparison
- When to use dimensionality reduction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_iris, load_digits, make_blobs
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline

---
## 1. The Curse of Dimensionality Revisited

As the number of features grows, several problems emerge:

- **Sparsity** — data points become increasingly far apart in high-dimensional space.
  Distance-based algorithms (K-Means, KNN, SVM with RBF) lose their discriminative
  power because all pairwise distances converge.
- **Overfitting** — more features means more parameters to estimate, increasing the
  risk of fitting noise rather than signal.
- **Computation** — training time and memory often scale with the number of features.
- **Visualisation** — humans can only see 2D or 3D projections.

Dimensionality reduction addresses all of these by projecting data into a
lower-dimensional space.

In [ ]:
# Demonstrate sparsity: average pairwise distance vs dimensionality
from sklearn.metrics import pairwise_distances

dims = [2, 5, 10, 50, 100, 500, 1000]
mean_dists = []
std_dists = []

rng = np.random.RandomState(42)
for d in dims:
    X_rnd = rng.randn(200, d)
    dists = pairwise_distances(X_rnd).ravel()
    dists = dists[dists > 0]  # exclude self-distances
    mean_dists.append(dists.mean())
    std_dists.append(dists.std())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(dims, mean_dists, 'o-', linewidth=2, color='steelblue')
axes[0].set_xlabel('Number of dimensions')
axes[0].set_ylabel('Mean pairwise distance')
axes[0].set_title('Distances grow with dimensionality')

# Ratio of std to mean (contrast)
contrast = [s / m for s, m in zip(std_dists, mean_dists)]
axes[1].plot(dims, contrast, 'o-', linewidth=2, color='darkorange')
axes[1].set_xlabel('Number of dimensions')
axes[1].set_ylabel('Std / Mean of pairwise distances')
axes[1].set_title('Distance contrast shrinks — all points look equally far')

plt.tight_layout()
plt.show()

As dimensionality increases, the ratio of standard deviation to mean distance
approaches zero. In other words, the "nearest" and "farthest" neighbours become
almost indistinguishable — bad news for distance-based algorithms.

---
## 2. Principal Component Analysis (PCA)

PCA is the most widely used dimensionality reduction technique. The idea:

1. **Centre** the data (subtract the mean of each feature).
2. Find the direction (axis) along which the data varies the most — this is the
   **first principal component** (PC1).
3. Find the next direction of maximum variance that is **orthogonal** to PC1 — this
   is PC2.
4. Continue until you have as many components as original features.
5. **Keep only the top K components** that capture most of the variance.

### Mathematical intuition

PCA computes the **eigenvectors** of the data's covariance matrix. Each eigenvector
defines a principal component direction, and its corresponding eigenvalue tells you
how much variance lies along that direction.

The projection onto the top K eigenvectors gives the best K-dimensional
approximation (in terms of least squared reconstruction error).

In [ ]:
# Visual intuition: PCA on 2D data
rng = np.random.RandomState(42)
X_2d = rng.randn(200, 2) @ np.array([[2, 0.5], [0.5, 0.8]])

# Fit PCA
pca_2d = PCA(n_components=2)
pca_2d.fit(X_2d)

# Get the principal component directions
mean = pca_2d.mean_
components = pca_2d.components_
explained = pca_2d.explained_variance_

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.4, edgecolors='k', s=30, c='steelblue')

# Draw the principal component arrows
for i, (comp, var) in enumerate(zip(components, explained)):
    # Scale arrow length by explained variance
    scale = 3 * np.sqrt(var)
    ax.annotate('', xy=mean + scale * comp, xytext=mean,
                arrowprops=dict(arrowstyle='->', linewidth=3,
                               color='red' if i == 0 else 'green'))
    ax.text(mean[0] + scale * comp[0] * 1.1,
            mean[1] + scale * comp[1] * 1.1,
            f'PC{i+1} ({var:.2f})', fontsize=12, fontweight='bold',
            color='red' if i == 0 else 'green')

ax.set_aspect('equal')
ax.set_title('PCA: principal components are directions of maximum variance')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()

The red arrow (PC1) points along the direction of greatest spread. The green arrow
(PC2) is orthogonal to it and captures the remaining variance. If we project all
points onto just the red line, we reduce from 2D to 1D while losing minimal
information.

---
## 3. PCA in Scikit-Learn

Using PCA is straightforward:

```python
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)
```

Key attributes after fitting:
- `pca.components_` — the principal component directions (each row is a component).
- `pca.explained_variance_` — the variance along each component.
- `pca.explained_variance_ratio_` — the fraction of total variance each component
  explains.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Original shape: {X_iris.shape}  (4 features)")
print(f"Features: {feature_names}")
print(f"Classes:  {list(target_names)}")

In [ ]:
# Scale first — PCA is sensitive to feature scales
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Fit PCA with all 4 components to inspect explained variance
pca_full = PCA(n_components=4)
pca_full.fit(X_iris_scaled)

print("Explained variance ratio per component:")
for i, ratio in enumerate(pca_full.explained_variance_ratio_):
    print(f"  PC{i+1}: {ratio:.4f} ({ratio*100:.1f}%)")
print(f"\nCumulative: {pca_full.explained_variance_ratio_.cumsum()}")

---
## 4. Explained Variance Ratio: Choosing the Number of Components

The explained variance ratio tells us how much information each component captures.
We want to keep enough components to retain a high fraction (e.g. 90--95%) of the
total variance.

Two common visualisations:
1. **Bar plot** — variance per component.
2. **Scree plot** (cumulative) — running total of explained variance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
axes[0].bar(range(1, 5), pca_full.explained_variance_ratio_,
            color='steelblue', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Variance explained by each component')
axes[0].set_xticks(range(1, 5))

# Cumulative scree plot
cumulative = pca_full.explained_variance_ratio_.cumsum()
axes[1].plot(range(1, 5), cumulative, 'o-', color='darkorange',
             linewidth=2, markersize=8)
axes[1].axhline(y=0.95, color='red', linestyle='--', alpha=0.7,
                label='95% threshold')
axes[1].fill_between(range(1, 5), cumulative, alpha=0.15, color='darkorange')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Scree plot — cumulative explained variance')
axes[1].set_xticks(range(1, 5))
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.tight_layout()
plt.show()

For the Iris dataset, the first two principal components capture roughly 95% of the
total variance. This means we can reduce from 4 features to 2 with minimal loss of
information — and now we can visualise the data in a 2D scatter plot.

---
## 5. PCA for Visualisation: Iris 4D to 2D

One of the most common uses of PCA is to project high-dimensional data down to 2D
for visualisation. Let us see how the three Iris species separate in the PC1-PC2
plane.

In [ ]:
pca_2 = PCA(n_components=2)
X_iris_2d = pca_2.fit_transform(X_iris_scaled)

fig, ax = plt.subplots(figsize=(9, 7))

for i, name in enumerate(target_names):
    mask = y_iris == i
    ax.scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1],
               label=name, s=60, edgecolors='k', alpha=0.8)

ax.set_xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('Iris dataset projected to 2D via PCA')
ax.legend(title='Species')
plt.tight_layout()
plt.show()

Even in just 2D, the three species are clearly separated. Setosa forms a distinct
cluster, while versicolor and virginica overlap slightly — consistent with what we
know about these species.

---
## 6. Understanding the Components: Feature Loadings

Each principal component is a linear combination of the original features. The
**loadings** (weights in `pca.components_`) tell us which original features
contribute most to each component.

In [ ]:
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=feature_names,
    columns=['PC1', 'PC2']
)
print("Feature loadings:")
print(loadings.round(4))

In [ ]:
# Biplot: scatter plot + loading arrows
fig, ax = plt.subplots(figsize=(9, 7))

# Scatter
for i, name in enumerate(target_names):
    mask = y_iris == i
    ax.scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1],
               label=name, s=50, edgecolors='k', alpha=0.6)

# Loading arrows
scale = 3
for j, feat in enumerate(feature_names):
    ax.annotate('', xy=(scale * pca_2.components_[0, j],
                        scale * pca_2.components_[1, j]),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', linewidth=2))
    ax.text(scale * pca_2.components_[0, j] * 1.15,
            scale * pca_2.components_[1, j] * 1.15,
            feat, fontsize=10, color='red', fontweight='bold')

ax.set_xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Biplot: PCA scores + feature loadings')
ax.legend(title='Species')
plt.tight_layout()
plt.show()

The arrows show how each original feature relates to the principal components.
Petal length and petal width point strongly along PC1, which is the main axis
separating the species.

---
## 7. PCA for 3D Visualisation

We can also project to 3D for an interactive-style view.

In [ ]:
pca_3 = PCA(n_components=3)
X_iris_3d = pca_3.fit_transform(X_iris_scaled)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for i, name in enumerate(target_names):
    mask = y_iris == i
    ax.scatter(X_iris_3d[mask, 0], X_iris_3d[mask, 1], X_iris_3d[mask, 2],
               label=name, s=50, edgecolors='k', alpha=0.7)

ax.set_xlabel(f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_zlabel(f'PC3 ({pca_3.explained_variance_ratio_[2]*100:.1f}%)')
ax.set_title('Iris dataset — PCA 3D projection')
ax.legend(title='Species')
plt.tight_layout()
plt.show()

---
## 8. PCA for Noise Reduction

An elegant application of PCA: if the signal lies in a low-dimensional subspace and
noise is spread across all dimensions, projecting onto the top components and then
reconstructing back to the original space effectively **denoises** the data.

$$\hat{X} = X_{\text{reduced}} \cdot W^T_{\text{top-K}} + \mu$$

The reconstruction discards the noisy minor components.

In [ ]:
# Create a clean 2D signal embedded in 10D with added noise
rng = np.random.RandomState(42)
t = np.linspace(0, 2 * np.pi, 200)
X_clean = np.column_stack([np.sin(t), np.cos(t)])  # circle in 2D

# Embed in 10D via a random projection
projection = rng.randn(2, 10)
X_10d_clean = X_clean @ projection

# Add noise
noise_level = 0.5
X_10d_noisy = X_10d_clean + noise_level * rng.randn(*X_10d_clean.shape)

# PCA reconstruction with 2 components
pca_denoise = PCA(n_components=2)
X_reduced = pca_denoise.fit_transform(X_10d_noisy)
X_reconstructed = pca_denoise.inverse_transform(X_reduced)

# Measure reconstruction error
error_noisy = np.mean((X_10d_noisy - X_10d_clean) ** 2)
error_denoised = np.mean((X_reconstructed - X_10d_clean) ** 2)
print(f"MSE before denoising:  {error_noisy:.4f}")
print(f"MSE after PCA denoise: {error_denoised:.4f}")
print(f"Noise reduction: {(1 - error_denoised / error_noisy) * 100:.1f}%")

In [ ]:
# Visualise in the PCA 2D space
X_clean_proj = pca_denoise.transform(X_10d_clean)
X_noisy_proj = pca_denoise.transform(X_10d_noisy)
X_recon_proj = pca_denoise.transform(X_reconstructed)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_clean_proj[:, 0], X_clean_proj[:, 1], s=10, c='steelblue')
axes[0].set_title('Clean signal (projected to PC space)')

axes[1].scatter(X_noisy_proj[:, 0], X_noisy_proj[:, 1], s=10, c='grey', alpha=0.5)
axes[1].set_title('Noisy signal (projected to PC space)')

axes[2].scatter(X_recon_proj[:, 0], X_recon_proj[:, 1], s=10, c='darkorange')
axes[2].set_title('PCA-denoised reconstruction')

for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_aspect('equal')

plt.suptitle('PCA for noise reduction: 10D signal denoised via 2-component reconstruction',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Choosing n_components Automatically

Instead of specifying an integer, you can pass a float between 0 and 1 to
`n_components`. PCA will then select the minimum number of components needed to
retain that fraction of the total variance.

In [ ]:
pca_auto = PCA(n_components=0.95)
X_iris_auto = pca_auto.fit_transform(X_iris_scaled)

print(f"Components chosen to retain 95% variance: {pca_auto.n_components_}")
print(f"Actual variance retained: {pca_auto.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Reduced shape: {X_iris_auto.shape}")

---
## 10. PCA on a Higher-Dimensional Dataset: Digits

The sklearn digits dataset has 64 features (8x8 pixel images). Let us see how
PCA compresses it.

In [ ]:
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Digits shape: {X_digits.shape}  (64 pixel features)")
print(f"Classes: {np.unique(y_digits)}")

# Show some sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(digits.images[i], cmap='Greys')
    ax.set_title(f'Label: {y_digits[i]}')
    ax.axis('off')
plt.suptitle('Sample digits (8x8 images)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# PCA on digits
scaler_d = StandardScaler()
X_digits_scaled = scaler_d.fit_transform(X_digits)

pca_digits = PCA().fit(X_digits_scaled)

fig, ax = plt.subplots(figsize=(9, 5))
cumvar = pca_digits.explained_variance_ratio_.cumsum()
ax.plot(range(1, 65), cumvar, 'o-', color='steelblue', markersize=3)
ax.axhline(y=0.90, color='red', linestyle='--', alpha=0.7, label='90%')
ax.axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95%')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('Scree plot — Digits dataset (64 features)')
ax.legend()

# Find the number of components for 90% and 95%
n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1
ax.axvline(x=n_90, color='red', linestyle=':', alpha=0.5)
ax.axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)
ax.text(n_90 + 1, 0.85, f'n={n_90}', fontsize=11, color='red')
ax.text(n_95 + 1, 0.92, f'n={n_95}', fontsize=11, color='orange')

plt.tight_layout()
plt.show()

print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")

In [ ]:
# PCA 2D projection of digits
pca_digits_2d = PCA(n_components=2)
X_digits_2d = pca_digits_2d.fit_transform(X_digits_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_digits_2d[:, 0], X_digits_2d[:, 1],
                     c=y_digits, cmap='tab10', s=10, alpha=0.7)
ax.set_xlabel(f'PC1 ({pca_digits_2d.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_digits_2d.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Digits dataset — PCA 2D projection')
plt.colorbar(scatter, label='Digit', ticks=range(10))
plt.tight_layout()
plt.show()

PCA in 2D shows some structure — certain digits form visible clusters — but
many classes overlap. PCA is a linear method and compresses by variance, not
by class separation. For better 2D visualisation of complex datasets, we need
a nonlinear approach: **t-SNE**.

---
## 11. t-SNE: Nonlinear Dimensionality Reduction

**t-SNE** (t-distributed Stochastic Neighbour Embedding) is designed specifically
for **visualisation**. It maps high-dimensional data to 2D (or 3D) by preserving
**local neighbourhood structure** — points that are close in the original space
should remain close in the embedding.

### How it works (intuition)
1. Compute pairwise similarities in the original space using a Gaussian kernel.
2. Compute pairwise similarities in the low-dimensional embedding using a
   Student-t distribution (heavier tails than Gaussian).
3. Minimise the KL divergence between the two similarity distributions via
   gradient descent.

### Key parameters
- `perplexity` (default 30) — roughly the number of effective nearest neighbours.
  Values between 5 and 50 are typical.
- `n_iter` — number of optimisation iterations (default 1000).
- `learning_rate` — step size for gradient descent (default 'auto').

### Important caveats
- t-SNE is for **visualisation only** — do not use the embedding as features for
  downstream models.
- The axes are meaningless — do not interpret distances between distant clusters.
- Results can vary with different random seeds and perplexity values.
- It is slow for large datasets — consider using PCA first to reduce to ~50D.

In [ ]:
# t-SNE on Iris
tsne_iris = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_iris_tsne = tsne_iris.fit_transform(X_iris_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA
for i, name in enumerate(target_names):
    mask = y_iris == i
    axes[0].scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1],
                    label=name, s=60, edgecolors='k', alpha=0.8)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('PCA on Iris')
axes[0].legend(title='Species')

# t-SNE
for i, name in enumerate(target_names):
    mask = y_iris == i
    axes[1].scatter(X_iris_tsne[mask, 0], X_iris_tsne[mask, 1],
                    label=name, s=60, edgecolors='k', alpha=0.8)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].set_title('t-SNE on Iris')
axes[1].legend(title='Species')

plt.suptitle('PCA vs t-SNE on the Iris dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

For the Iris dataset both PCA and t-SNE give nice separations. The real power of
t-SNE shows on more complex, higher-dimensional data like the digits dataset.

---
## 12. t-SNE on Digits: Where It Really Shines

In [ ]:
# First reduce to 30D with PCA (recommended for speed and stability)
pca_pre = PCA(n_components=30, random_state=42)
X_digits_pca30 = pca_pre.fit_transform(X_digits_scaled)

tsne_digits = TSNE(n_components=2, perplexity=30, random_state=42,
                    n_iter=1500, learning_rate='auto')
X_digits_tsne = tsne_digits.fit_transform(X_digits_pca30)

print(f"t-SNE embedding shape: {X_digits_tsne.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA 2D
sc1 = axes[0].scatter(X_digits_2d[:, 0], X_digits_2d[:, 1],
                       c=y_digits, cmap='tab10', s=8, alpha=0.7)
axes[0].set_title('PCA 2D projection of Digits')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[0], label='Digit', ticks=range(10))

# t-SNE 2D
sc2 = axes[1].scatter(X_digits_tsne[:, 0], X_digits_tsne[:, 1],
                       c=y_digits, cmap='tab10', s=8, alpha=0.7)
axes[1].set_title('t-SNE 2D embedding of Digits')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(sc2, ax=axes[1], label='Digit', ticks=range(10))

plt.suptitle('PCA vs t-SNE on the Digits dataset (64D -> 2D)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

The difference is dramatic. PCA squashes the 10 digit classes into overlapping blobs,
while t-SNE produces clearly separated islands for each digit. This is because t-SNE
preserves local structure (which digit looks like which) rather than global variance.

---
## 13. Effect of Perplexity on t-SNE

The `perplexity` parameter controls the balance between local and global structure.
Let us see how different values affect the embedding.

In [ ]:
perplexities = [5, 15, 30, 50, 100]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for ax, perp in zip(axes, perplexities):
    tsne_p = TSNE(n_components=2, perplexity=perp, random_state=42,
                   n_iter=1000, learning_rate='auto')
    X_emb = tsne_p.fit_transform(X_digits_pca30)
    ax.scatter(X_emb[:, 0], X_emb[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f'perplexity={perp}')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('t-SNE on Digits: effect of perplexity', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

- **Low perplexity** (5) — very tight, fragmented clusters. Focuses on very local
  structure.
- **Medium perplexity** (30) — typically a good default. Clear, well-separated clusters.
- **High perplexity** (100) — clusters start to merge as the algorithm considers
  broader neighbourhoods.

Always try a few perplexity values to ensure the pattern is stable, not an artefact.

---
## 14. PCA vs t-SNE: Summary Comparison

| Aspect | PCA | t-SNE |
|--------|-----|-------|
| **Type** | Linear | Nonlinear |
| **Preserves** | Global variance | Local neighbourhood structure |
| **Use cases** | Preprocessing, noise reduction, visualisation | Visualisation only |
| **Speed** | Very fast | Slow (esp. on large data) |
| **Deterministic** | Yes | No (depends on random seed) |
| **Invertible** | Yes (`inverse_transform`) | No |
| **Interpretable axes** | Yes (loadings) | No |
| **Works as preprocessing** | Yes | Not recommended |
| **Typical output dims** | Any (but often 2--50) | 2 or 3 |

**Best practice:** Use PCA for feature reduction and preprocessing. Use t-SNE
(often after PCA to ~50D) for 2D visualisation of complex high-dimensional data.

---
## 15. When to Use Dimensionality Reduction

Dimensionality reduction is valuable in several scenarios:

1. **Visualisation** — project to 2D/3D to understand cluster structure, class
   separation, or data quality issues.
2. **Preprocessing** — reduce features before a slow algorithm (e.g. KNN on
   high-dimensional data).
3. **Noise reduction** — PCA reconstruction removes noise from minor components.
4. **Multicollinearity** — PCA components are orthogonal by construction,
   eliminating correlated features.
5. **Storage and speed** — fewer features mean smaller models and faster training.

**When NOT to reduce:**
- When you need interpretable features (PCA components are linear combinations,
  harder to explain than raw features).
- When the number of features is already small.
- When a tree-based model (random forest, gradient boosting) handles the full
  feature set well — they are inherently robust to irrelevant features.

---
## 16. Summary

| Concept | Key point |
|---------|----------|
| Curse of dimensionality | Distances lose meaning in high dimensions |
| PCA | Linear: projects onto directions of maximum variance |
| Explained variance ratio | Fraction of total variance captured per component |
| Scree plot | Cumulative variance vs n_components — pick the "elbow" or a threshold |
| PCA for visualisation | Project to 2D/3D for scatter plots |
| PCA for denoising | Reconstruct from top components to remove noise |
| t-SNE | Nonlinear: preserves local neighbourhood for 2D visualisation |
| Perplexity | t-SNE's key parameter; try 5--50; affects cluster tightness |
| PCA vs t-SNE | PCA for preprocessing; t-SNE for visualisation |

---

**This concludes Chapter 14: Unsupervised Learning.** We have covered the three pillars
of unsupervised learning: clustering (K-Means, hierarchical, DBSCAN) and dimensionality
reduction (PCA, t-SNE). Together, these tools let you discover structure, compress
features, and visualise complex datasets — all without a single label.